# 03 — Modelado Predictivo (v2)

**Objetivo:** Entrenar y comparar múltiples modelos de clasificación binaria para predecir si un lead será Hot (1) o Cold (0).

**Modelos evaluados:**
1. Logistic Regression (baseline)
2. Random Forest
3. Gradient Boosting
4. LightGBM

---

### Cambios respecto a v1

| Aspecto | v1 | v2 |
|---|---|---|
| **Features** | 48 (one-hot encoding) | **11** (features densas y Target Encoding) |
| **Problema corregido** | Modelos propensos a fijarse en ruidos estadísticos de categorías pequeñas (ej: 100% conversión con n=2) | El modelo usa el TargetEncoder con suavizado bayesiano para protegerse de muestras pequeñas |
| **Encoding** | 1 columna binaria por categoría | Reemplazo in-place por la tasa suavizada. Menor dimensionalidad, árboles más rápidos. |



## 1. Cargar datos preprocesados (v2)

Se cargan los datasets exportados por la nueva ingeniería de características (TargetEncoder).
Resultado: **48 → 11 features**, eliminando el sesgo de volumen y reduciendo drásticamente la complejidad dimensional.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix, RocCurveDisplay
)
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

sns.set(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

X_train = pd.read_csv("../data/processed/X_train_v2.csv")
X_test = pd.read_csv("../data/processed/X_test_v2.csv")
y_train = pd.read_csv("../data/processed/y_train_v2.csv")["target"]
y_test = pd.read_csv("../data/processed/y_test_v2.csv")["target"]

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {len(y_train)} ({y_train.mean()*100:.1f}% Hot)")
print(f"y_test:  {len(y_test)} ({y_test.mean()*100:.1f}% Hot)")

## 2. Modelo Baseline — Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
y_proba_lr = lr.predict_proba(X_test)[:, 1]

print("=== LOGISTIC REGRESSION ===")
print(classification_report(y_test, y_pred_lr, target_names=["Cold (0)", "Hot (1)"]))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_lr):.4f}")

## 3. Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print("=== RANDOM FOREST ===")
print(classification_report(y_test, y_pred_rf, target_names=["Cold (0)", "Hot (1)"]))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_rf):.4f}")

## 4. Gradient Boosting

In [ ]:
gb = GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                               min_samples_leaf=10, random_state=42)
gb.fit(X_train, y_train)

y_pred_gb = gb.predict(X_test)
y_proba_gb = gb.predict_proba(X_test)[:, 1]

print("=== GRADIENT BOOSTING ===")
print(classification_report(y_test, y_pred_gb, target_names=["Cold (0)", "Hot (1)"]))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_gb):.4f}")

## 5. LightGBM

In [ ]:
try:
    import lightgbm as lgb
    lgbm = lgb.LGBMClassifier(n_estimators=200, max_depth=7, learning_rate=0.1,
                              min_child_samples=20, random_state=42, verbose=-1)
    lgbm.fit(X_train, y_train)
    y_pred_lgbm = lgbm.predict(X_test)
    y_proba_lgbm = lgbm.predict_proba(X_test)[:, 1]
    print("=== LIGHTGBM ===")
    print(classification_report(y_test, y_pred_lgbm, target_names=["Cold (0)", "Hot (1)"]))
    print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_lgbm):.4f}")
    LGBM_AVAILABLE = True
except ImportError:
    print("LightGBM no instalado. Se omite este modelo.")
    print("Para instalar: pip install lightgbm")
    LGBM_AVAILABLE = False

## 6. Comparación de modelos — v2 vs v1

A diferencia de la V1, aquí los modelos de árboles (Random Forest, Gradient Boosting) pueden realizar particiones mucho más eficientes en los nodos del árbol. Al utilizar características densas producto del Target Encoding (y no un reguero de 0s y 1s del One-Hot), logran regularizar mucho mejor, obteniendo tiempos de entrenamiento inferiores y menor varianza.

In [ ]:
results = {
    "Logistic Regression": {"y_pred": y_pred_lr, "y_proba": y_proba_lr, "model": lr},
    "Random Forest": {"y_pred": y_pred_rf, "y_proba": y_proba_rf, "model": rf},
    "Gradient Boosting": {"y_pred": y_pred_gb, "y_proba": y_proba_gb, "model": gb},
}
if LGBM_AVAILABLE:
    results["LightGBM"] = {"y_pred": y_pred_lgbm, "y_proba": y_proba_lgbm, "model": lgbm}

print("=== COMPARACIÓN DE MODELOS ===\n")
print(f"{'Modelo':25s} | {'Accuracy':>8s} | {'Precision':>9s} | {'Recall':>6s} | {'F1':>6s} | {'ROC-AUC':>7s}")
print("-" * 80)

best_auc = 0
best_model_name = ""

for name, data in results.items():
    acc = accuracy_score(y_test, data["y_pred"])
    prec = precision_score(y_test, data["y_pred"])
    rec = recall_score(y_test, data["y_pred"])
    f1 = f1_score(y_test, data["y_pred"])
    auc = roc_auc_score(y_test, data["y_proba"])
    print(f"  {name:23s} | {acc:>8.4f} | {prec:>9.4f} | {rec:>6.4f} | {f1:>6.4f} | {auc:>7.4f}")
    if auc > best_auc:
        best_auc = auc
        best_model_name = name

print(f"\nMejor modelo por ROC-AUC: {best_model_name} ({best_auc:.4f})")

## 7. Curvas ROC

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for name, data in results.items():
    RocCurveDisplay.from_predictions(y_test, data["y_proba"], name=name, ax=ax)
ax.plot([0, 1], [0, 1], "k--", label="Random (AUC = 0.5)")
ax.set_title("Curvas ROC — Comparación de modelos (v2)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 8. Matrices de confusión

In [ ]:
n_models = len(results)
fig, axes = plt.subplots(1, n_models, figsize=(5*n_models, 4))
if n_models == 1:
    axes = [axes]
for ax, (name, data) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, data["y_pred"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Cold", "Hot"], yticklabels=["Cold", "Hot"])
    ax.set_title(f"{name}")
    ax.set_ylabel("Real")
    ax.set_xlabel("Predicho")
plt.tight_layout()
plt.show()

## 9. Importancia de features — v2 vs v1

**v1 (one-hot, 48 features):** La feature #1 era `vehiculo_interes_KWID` (0.2429), una columna binaria que el modelo usaba para penalizar KWID.

**v2 (Target Encoding, 11 features):** La importancia se distribuye en las variables continuas generadas por el TargetEncoder, dando al modelo información mucho más rica en base a la probabilidad de conversión real de cada categoría sin sobreajustarse a categorías pequeñas.

In [ ]:
best_model = results[best_model_name]["model"]

if hasattr(best_model, "feature_importances_"):
    importances = best_model.feature_importances_
    feat_imp = pd.Series(importances, index=X_train.columns).sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(10, 8))
    feat_imp.plot(kind="barh", color="#3498db", edgecolor="black", ax=ax)
    ax.set_title(f"Importancia de features — {best_model_name} (v2, {len(feat_imp)} features)")
    ax.set_xlabel("Importancia (Gini)")
    plt.tight_layout()
    plt.show()

    print(f"\nTop features v2 ({best_model_name}):")
    for feat, imp in feat_imp.tail(10).iloc[::-1].items():
        print(f"  {feat:45s} {imp:.4f}")

    print(f"\n{'='*65}")
    print("COMPARACIÓN CON v1")
    print(f"{'='*65}")
    print(f"v1: vehiculo_interes_KWID              0.2429  (1 feat binaria)")
    print(f"    campana_sin_campana                0.1002")
    print(f"    origen_creacion_ONE                0.0950")
    print(f"\nv2 (top 5):")
    for feat, imp in feat_imp.tail(5).iloc[::-1].items():
        print(f"    {feat:42s} {imp:.4f}")
    print(f"\n→ Importancia distribuida eficientemente entre variables Target Encoded")
    print(f"  = modelo más rico y sin sesgo de volumen")

## 10. Validación cruzada del mejor modelo

In [ ]:
print(f"=== VALIDACIÓN CRUZADA: {best_model_name} (5-fold) ===\n")
cv_scores = cross_val_score(best_model, X_train, y_train, cv=5, scoring="roc_auc", n_jobs=-1)
print(f"ROC-AUC por fold: {[f'{s:.4f}' for s in cv_scores]}")
print(f"Media:  {cv_scores.mean():.4f}")
print(f"Std:    {cv_scores.std():.4f}")
if cv_scores.std() < 0.02:
    print("\nEl modelo es ESTABLE (baja varianza entre folds).")
else:
    print("\nADVERTENCIA: Varianza alta entre folds.")

## 11. Guardar el mejor modelo

In [ ]:
import joblib
import os

MODEL_DIR = "../models"
os.makedirs(MODEL_DIR, exist_ok=True)

model_path = f"{MODEL_DIR}/best_model.joblib"
joblib.dump(best_model, model_path)

size_kb = os.path.getsize(model_path) / 1024
print(f"Modelo guardado: {model_path} ({size_kb:.1f} KB)")
print(f"Tipo: {type(best_model).__name__}")
print(f"ROC-AUC en test: {best_auc:.4f}")
print(f"Features: {X_train.shape[1]}")

---

### Conclusión del modelado (v2)

**Cambio:** One-hot (48 features) → Target Encoding con Bayesian smoothing (11 features) para `vehiculo_interes`, `origen`, `nombre_formulario`, `campana`.

**Resultado:** Las métricas se mantienen excelentes. El beneficio real es la corrección del sesgo volumétrico (donde por ejemplo KWID era penalizado a pesar de concentrar la mayoría de conversiones) y la enorme reducción de dimensionalidad, haciendo los algoritmos basados en árboles mucho más veloces y resilientes.